# Breast Cancer Classification — Logistic Regression

An educational machine-learning project using the Wisconsin Diagnostic Breast Cancer dataset.

**Target:** `diagnosis` (`B` = benign, `M` = malignant). This project is for learning only and is not a medical diagnostic tool.

Keep this notebook in the same folder as `breast-cancer.csv`.

In [ ]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("breast-cancer.csv")
df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())
df.info()

In [ ]:
print("Missing values:")
display(df.isnull().sum().to_frame("missing_count"))
print("Diagnosis counts:")
print(df["diagnosis"].value_counts())

In [ ]:
# Remove the record identifier; it is not a tumor measurement.
model_df = df.drop(columns=["id"], errors="ignore").copy()
X = model_df.drop(columns=["diagnosis"])
y = model_df["diagnosis"].map({"B": 0, "M": 1})

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Labels: 0 = benign, 1 = malignant")

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=2000, random_state=42)
model.fit(X_train_scaled, y_train)
print("Model training completed!")

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_pred = model.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Benign", "Malignant"]))
print("Confusion matrix (rows=actual, columns=predicted; order 0, 1):")
print(confusion_matrix(y_test, y_pred))

In [ ]:
# Example prediction for one held-out row
sample = X_test.iloc[[0]]
sample_scaled = scaler.transform(sample)
prediction = int(model.predict(sample_scaled)[0])
probability = float(model.predict_proba(sample_scaled)[0][1])
print("Model output:", "Malignant" if prediction == 1 else "Benign")
print(f"Model-estimated malignant probability: {probability:.1%}")
print("Educational use only; not for medical diagnosis.")

In [ ]:
# Save model and scaler so app.py can reuse them without retraining.
import joblib

joblib.dump(model, "breast_cancer_logistic_model.pkl")
joblib.dump(scaler, "breast_cancer_scaler.pkl")
print("Saved model and scaler files.")

## Next steps

- Run `pip install -r requirements.txt` in your environment.
- Run `streamlit run app.py` to launch the demo interface.
- Evaluate recall and false negatives carefully; accuracy alone is not sufficient.
- Do not use this educational model to make medical decisions.